Python commands


In [0]:
df=spark.sql('select * from gizmobox.bronze.payments');
display(df)

In [0]:
df2=spark.sql('SELECT * FROM json.`/Volumes/gizmobox/landing/operational_data/customers/customers_2024_10.json`');
display(df2)
# df2=spark.read.json('/Volumes/gizmobox/landing/operational_data/customers').display()
#df3=spark.read.format('json').load('/Volumes/gizmobox/landing/operational_data/customers')


In [0]:
df2=spark.read.json('/Volumes/gizmobox/landing/operational_data/customers')
df3=spark.read.format('json').load('/Volumes/gizmobox/landing/operational_data/customers');
display(df3)

In [0]:
spark.table('gizmobox.bronze.v_customers').display()

In [0]:
# add path meta data 
df_with_metadat=df3.select ('_metadata.file_path','*');
display(df_with_metadat)

In [0]:
# write ad create tables 
df_with_metadat.write.format('delta').mode('overwrite').saveAsTable('gizmobox.bronze.py_customers')

In [0]:
%sql
drop table gizmobox.bronze.py_customers


In [0]:
df_with_metadat.writeTo('gizmobox.bronze.py_customers').createOrReplace();
spark.sql('select * from gizmobox.bronze.py_customers').display()

In [0]:
%sql
select * from gizmobox.silver.py_customers;


In [0]:
df_orders=spark.read.text('/Volumes/gizmobox/landing/operational_data/orders')
display(df_orders);
df_orders.writeTo('gizmobox.bronze.py_orders').createOrReplace();
spark.sql('select * from gizmobox.bronze.py_orders')


In [0]:
df_mem=spark.read.format('binaryFile').load('/Volumes/gizmobox/landing/operational_data/memberships/*/*.png')
display(df_mem);
df_mem.writeTo('gizmobox.bronze.py_membership').createOrReplace();
spark.sql('select * from gizmobox.bronze.py_membership')

In [0]:
%sql

SELECT * 
FROM csv.`/Volumes/gizmobox/landing/operational_data/addresses`;


In [0]:
#df_adr=spark.sql('SELECT * FROM csv.`/Volumes/gizmobox/landing/operational_data/addresses`', header=True,delimiter=',')
df_adr=spark.read.format('csv').option('header','true').option('delimiter','\t').load('/Volumes/gizmobox/landing/operational_data/addresses')
display(df_adr)
df_adr.writeTo('gizmobox.bronze.py_address').createOrReplace();
spark.sql('select * from gizmobox.bronze.py_address')


In [0]:
%sql
select * from gizmobox.bronze.payments
-- the source csv file does not have a hader in this case 

In [0]:
%fs ls  'abfss://gizmobox@daecourseexdl.dfs.core.windows.net/landing/external_data/payments'


In [0]:
payments_schema = 'payment_id INTEGER, order_id INTEGER, payment_timestamp TIMESTAMP, payment_status INTEGER, payment_method STRING';
# or 
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType

py_payments_schema = StructType([
    StructField("payment_id", IntegerType()),
    StructField("order_id", IntegerType()),
    StructField("payment_timestamp", TimestampType()),
    StructField("payment_status", IntegerType()),
    StructField("payment_method", StringType())
])

In [0]:
df = (
    spark.read.format('csv')
         .option('delimiter', ',')
         .schema(py_payments_schema) 
         .load('abfss://gizmobox@daecourseexdl.dfs.core.windows.net/landing/external_data/payments')
)
display(df)
# or payments_schema just two ways of figuring schema for csv files without header

extract data from sql tables 
- JDBC